In [ ]:
!pip install pyspark==3.5.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.9/317.9 MB 14.5 MB/s  0:00:22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.9-py2.py3-none-any.whl size=318391123 sha256=3d5e2b54b31e3ff1403139bb1bc0d816ca38c0451e24bf42ee1283745a46da71
  Stored in directory: /root/.cache/pip/wheels/c3/63/84/bf128a5a98cb9e9a8a263b51949e36867809b2b5bd855ac5e7
Successfully built pyspark
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.4
    Uninstalling pyspark-4.0.4:
      Successfully uninstalled pyspark-4.0.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.9 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_sales.csv to cleaned_sales.csv


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, date_format, round as spark_round
spark = (
    SparkSession.builder
    .appName("RetailSalesWeek3Analysis")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv(
    "/content/cleaned_sales.csv",
    header=True,
    inferSchema=True
)
print("Total records:", df.count())
df.printSchema()
df.show(5, truncate=False)

Total records: 21
root
 |-- sale_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- gross_revenue: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_revenue: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- profit_margin_percent: double (nullable = true)

+-------+----------+----------+--------------+-----------+-------+--------+---------------------+------+-----------+--------+----------+----------------+-------------+---

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, avg
product_summary = (
    df.groupBy("product_id", "product_name")
    .agg(
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_sum("net_revenue").alias("total_net_revenue"),
        spark_sum("profit").alias("total_profit")
    )
)
# Calculate average quantity sold across products
average_quantity = (
    product_summary
    .agg(avg("total_quantity_sold").alias("average_quantity"))
    .first()["average_quantity"]
)
# Identify products selling below the average
underperforming_products = (
    product_summary
    .filter(col("total_quantity_sold") < average_quantity)
    .orderBy(col("total_quantity_sold").asc())
)
print("Average quantity sold per product:", average_quantity)
underperforming_products.show(truncate=False)

Average quantity sold per product: 13.2
+----------+------------+-------------------+-----------------+------------+
|product_id|product_name|total_quantity_sold|total_net_revenue|total_profit|
+----------+------------+-------------------+-----------------+------------+
|8         |Desk        |3                  |29000.0          |8000.0      |
|4         |Smartphone  |5                  |119750.0         |29750.0     |
|1         |Laptop      |5                  |282000.0         |57000.0     |
|10        |Printer     |5                  |69900.0          |14900.0     |
|6         |Monitor     |6                  |69600.0          |18600.0     |
|7         |Office Chair|7                  |51000.0          |16000.0     |
|3         |Keyboard    |12                 |17850.0          |9450.0      |
+----------+------------+-------------------+-----------------+------------+



In [ ]:
underperforming_products.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/underperforming_products")

In [ ]:
import glob
from google.colab import files
csv_files = glob.glob("/content/underperforming_products/part-*.csv")
files.download(csv_files[0])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, avg, date_format, round as spark_round
# Extract year and month from the sale date
monthly_store_revenue = (
    df.withColumn(
        "month",
        date_format(col("sale_date"), "yyyy-MM")
    )
    .groupBy("store_id", "store_name", "month")
    .agg(
        spark_sum("net_revenue").alias("monthly_revenue")
    )
)
print("MONTHLY REVENUE BY STORE")
monthly_store_revenue.orderBy(
    "store_id", "month"
).show(truncate=False)

MONTHLY REVENUE BY STORE
+--------+---------------------+-------+---------------+
|store_id|store_name           |month  |monthly_revenue|
+--------+---------------------+-------+---------------+
|1       |Chennai Central Store|2026-07|122000.0       |
|1       |Chennai Central Store|2026-09|12000.0        |
|1       |Chennai Central Store|2026-10|136500.0       |
|2       |Bengaluru City Store |2026-07|78600.0        |
|2       |Bengaluru City Store |2026-08|60000.0        |
|2       |Bengaluru City Store |2026-09|45600.0        |
|2       |Bengaluru City Store |2026-10|9000.0         |
|3       |Mumbai Retail Store  |2026-07|38400.0        |
|3       |Mumbai Retail Store  |2026-08|11400.0        |
|3       |Mumbai Retail Store  |2026-09|22500.0        |
|4       |Delhi Main Store     |2026-08|48500.0        |
|4       |Delhi Main Store     |2026-09|10500.0        |
|5       |Hyderabad Tech Store |2026-08|47400.0        |
|5       |Hyderabad Tech Store |2026-09|57500.0        |
+-----

In [ ]:
average_monthly_store_revenue = (
    monthly_store_revenue
    .groupBy("store_id", "store_name")
    .agg(
        spark_round(
            avg("monthly_revenue"), 2
        ).alias("average_monthly_revenue")
    )
    .orderBy(col("average_monthly_revenue").desc())
)
print("AVERAGE MONTHLY REVENUE BY STORE")
average_monthly_store_revenue.show(truncate=False)

AVERAGE MONTHLY REVENUE BY STORE
+--------+---------------------+-----------------------+
|store_id|store_name           |average_monthly_revenue|
+--------+---------------------+-----------------------+
|1       |Chennai Central Store|90166.67               |
|5       |Hyderabad Tech Store |52450.0                |
|2       |Bengaluru City Store |48300.0                |
|4       |Delhi Main Store     |29500.0                |
|3       |Mumbai Retail Store  |24100.0                |
+--------+---------------------+-----------------------+



In [ ]:
average_monthly_store_revenue.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/average_monthly_store_revenue")

In [ ]:
import glob
from google.colab import files
csv_files = glob.glob(
    "/content/average_monthly_store_revenue/part-*.csv"
)
files.download(csv_files[0])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>